# 第14章：CAPM与资产定价

先用四行构造收益逐项计算均值、协方差与beta，核对0.5和1.5，再绘制两幅构造CAPM图。假定的无风险期望收益与市场期望收益分别为2%和8%，不要把它们同四行样本的实现均值混为一谈。French月频回归默认禁用，保留的2010–2019年说明只属于历史描述，不提供PIT版本或交易预测。

[本语言逐本运行模式与准备条件](../NOTEBOOK_GUIDE.md)。本阶段只制作语言层，便携环境与执行在后续隔离冷启动核验。

In [ ]:
from pathlib import Path
import sys, os, uuid
ROOT = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
             if (p / "pyproject.toml").is_file() and (p / "src/quant_course").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Start inside the reader package tree")
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "code"))
os.environ["MPLBACKEND"] = "Agg"
RUN_DIR = ROOT / "runs" / ("ch14_asset_pricing_" + uuid.uuid4().hex)
RUN_DIR.mkdir(parents=True, exist_ok=False)
os.chdir(ROOT)
# French branches remain disabled until data rights and execution authorization are resolved.
RUN_FRENCH = False
print("CONSTRUCTED teaching route; French market branches SKIPPED; outputs:", RUN_DIR)

from pathlib import Path
import numpy as np
import importlib.util
import sys


from quant_course.asset_pricing import (
    beta_from_covariance,
    capm_expected_return,
    constructed_two_asset_sample,
    fit_capm_excess,
)

# 复用 CLI 中经过 manifest SHA 和连续月频校验的唯一数据 loader。
_spec = importlib.util.spec_from_file_location(
    "ch14_asset_pricing_cli", ROOT / "code/ch14_asset_pricing.py"
)
ch14 = importlib.util.module_from_spec(_spec)
sys.modules[_spec.name] = ch14
_spec.loader.exec_module(ch14)


## 构造例：均值、协方差、beta 与残差

四行数据由本章代码显式构造，等权只服务于手算核对。样本协方差使用分母 `n−1`，矩阵顺序为市场超额收益、Asset A 超额收益、Asset B 超额收益。

In [ ]:
sample = constructed_two_asset_sample()
returns = np.column_stack([
    sample["Mkt-RF"],
    sample["Asset_A"] - sample["RF"],
    sample["Asset_B"] - sample["RF"],
])
mean_vector = returns.mean(axis=0)
covariance = np.cov(returns, rowvar=False, ddof=1)
fit_a = fit_capm_excess(returns[:, 1], returns[:, 0])
fit_b = fit_capm_excess(returns[:, 2], returns[:, 0])

print("rows are constructed; returns are decimals per single period")
print("mean [Mkt-RF, A-RF, B-RF]:", mean_vector)
print("sample covariance, ddof=1:\n", covariance)
print("beta A / B:", fit_a.beta, fit_b.beta)
print("residuals A:", fit_a.residuals)
print("residuals B:", fit_b.residuals)
assert abs(fit_a.beta - 0.5) < 1e-12
assert abs(fit_b.beta - 1.5) < 1e-12


In [ ]:
# 在另一组显式构造的期望输入下，SML 给出的模型期望收益。
rf_expected = 0.02
market_expected = 0.08
for beta in (0.5, 1.5):
    print(beta, capm_expected_return(rf_expected, beta, market_expected))

ch14.make_sml_cml_figure(RUN_DIR / "ch14_sml_cml.png")
ch14.make_beta_scatter_figure(RUN_DIR / "ch14_beta_scatter.png")
from IPython.display import Image, display
display(Image(filename=str(RUN_DIR / "ch14_sml_cml.png")))
display(Image(filename=str(RUN_DIR / "ch14_beta_scatter.png")))


图中曲线是构造输入下的 CAPM 关系，散点是四行记录的实现均值。即使两者不同，也不能把构造样例解释为模型拒绝或投资机会。

## 默认禁用的French NoDur月频回归

历史加载器若经许可启用，会先按manifest核对处理文件的SHA-256，连接NoDur、Mkt-RF和RF，并要求2010年1月至2019年12月恰有120个连续月份。Mkt-RF已经是超额收益，资产总收益只减同月RF一次。接口接受月度PeriodIndex，HAC maxlags=5以月度观察行计。默认不调用该加载器，也不读取随包市场文件。

In [ ]:
if RUN_FRENCH:
    monthly, industry_manifest, factor_manifest = ch14.load_french_sample()
    report, monthly_fit = ch14.french_report(monthly, industry_manifest, factor_manifest)
    print({key: report[key] for key in (
        "sample_start", "sample_end", "frequency", "n_months",
        "industry_processed_sha256", "factor_processed_sha256",
        "alpha_monthly", "alpha_hac_se_monthly", "alpha_hac_95ci_monthly",
        "beta", "r_squared", "hac", "scope",
    )})
    assert len(monthly) == 120
    assert str(monthly.index[0]) == "2010-01"
    assert str(monthly.index[-1]) == "2019-12"
    assert monthly.index.freqstr == "M"
else:
    print("SKIPPED French branch: permissions unknown; no bundled data; historical empirical results not reproduced")


In [ ]:
if RUN_FRENCH:
    ch14.make_french_regression_figure(
        monthly, monthly_fit, RUN_DIR / "ch14_nodur_regression_residuals.png"
    )
    display(Image(filename=str(RUN_DIR / "ch14_nodur_regression_residuals.png")))
else:
    print("SKIPPED French branch: permissions unknown; no bundled data; historical empirical results not reproduced")


Alpha是指定窗口、市场代理和模型下的月度OLS截距估计，不是总体期望alpha的直接观测，更不是下月或成本后策略收益。历史French样本属于发布时的修订快照，没有逐月PIT版本，也未定义信号或成交时点。截距区间跨零不证明CAPM成立，不跨零也不单独证明可实施套利。默认构造例不产生这些历史回归结论。